# Activity 3 – Preparing for Data Analysis
**Course:** Advanced Programming – Week 4  
**Author:** Sebastian Diaz  

Practice identifying and applying effective techniques for cleaning data in preparation for data analysis.

In [ ]:
import pandas as pd
import numpy as np
import hashlib

---
## Exercise 1 – Data Cleaning: PeoplesFavourites Dataset

**File:** `Activity_3_Exercise_1_PeoplesFavourites-1.csv`

**Tasks:**
1. Explore the data to identify cleaning issues
2. Standardise null value representation
3. Remove records with missing first name, last name, or email
4. Anonymise by replacing names with a unique identifier
5. Export cleaned data to a new file

### Step 1: Explore the Data

In [ ]:
# Load the raw data
df_raw = pd.read_csv('Activity_3_Exercise_1_PeoplesFavourites-1.csv')

print("Shape:", df_raw.shape)
print("\nColumn names:")
print(df_raw.columns.tolist())
print("\nFirst 5 rows:")
df_raw.head()

In [ ]:
# Check data types and non-null counts
print(df_raw.info())

In [ ]:
# Check for duplicate rows
print("Duplicate rows:", df_raw.duplicated().sum())

# Check unique null-like values in a sample of columns to understand inconsistencies
sample_cols = ['First Name', 'Last Name', 'Email', 'Interests', 'Food']
for col in sample_cols:
    if col in df_raw.columns:
        unique_vals = df_raw[col].unique()
        # Show potentially null-like values
        suspicious = [v for v in unique_vals if str(v).strip().lower() in
                      ['none', 'null', '0', '', 'nan']]
        print(f"  '{col}' — null-like values found: {suspicious}")

**Observations from exploration:**
- Several columns use mixed null representations: `none`, `null`, `0`, and empty strings.
- The dataset contains both complete and partial records.
- `First Name`, `Last Name`, and `Email` are critical identifying fields that must be present before anonymisation.
- There are records with no first name, last name, and no email (e.g., rows where the email follows the pattern but names are absent).

### Step 2: Standardise Null Values

In [ ]:
# Work on a copy to preserve the original
df = df_raw.copy()

# Define all symbols used to represent null/missing data in this dataset
# Note: '0' as income or a valid 0 value would need careful handling;
#        here '0' in text columns is clearly a null placeholder
null_symbols = ['none', 'null', '0', '', ' ']

# Replace all null-like strings with np.nan (case-insensitive)
# First strip whitespace, then replace
df = df.apply(lambda col: col.map(
    lambda x: np.nan if str(x).strip().lower() in null_symbols else x
))

# Also handle the integer 0 (loaded as numeric by pandas in Income column)
# Replace 0 in Income with NaN since 0 income is a null placeholder here
if 'Income' in df.columns:
    df['Income'] = df['Income'].replace(0, np.nan)

print("Null counts after standardisation:")
print(df.isnull().sum())

### Step 3: Remove Records with Missing Critical Fields

In [ ]:
rows_before = len(df)

# Remove rows where First Name, Last Name, or Email are missing
critical = ['First Name', 'Last Name', 'Email']
df = df.dropna(subset=critical)

rows_after = len(df)
print(f"Rows before removal : {rows_before}")
print(f"Rows after removal  : {rows_after}")
print(f"Rows removed        : {rows_before - rows_after}")

### Step 4: Remove Duplicates

In [ ]:
rows_before_dedup = len(df)
df = df.drop_duplicates()
print(f"Duplicate rows removed: {rows_before_dedup - len(df)}")
print(f"Rows remaining: {len(df)}")

### Step 5: Anonymise – Replace Names with Unique Identifiers

In [ ]:
# Reset index after drops
df = df.reset_index(drop=True)

# Generate a unique customer ID based on a hash of the email
# This is reproducible (same email → same ID) and non-reversible
def generate_id(email):
    return 'CUST_' + hashlib.sha256(str(email).encode()).hexdigest()[:8].upper()

# Insert the customer ID as the first column
df.insert(0, 'Customer_ID', df['Email'].apply(generate_id))

# Remove the identifying columns
df = df.drop(columns=['First Name', 'Last Name', 'Email'])

print("Anonymised DataFrame — first 5 rows:")
print(df.head())
print("\nColumns:")
print(df.columns.tolist())

### Step 6: Export Cleaned Data

In [ ]:
output_path = 'Activity_3_Exercise_1_PeoplesFavourites_Cleaned.csv'
df.to_csv(output_path, index=False)
print(f"Cleaned data exported to: {output_path}")
print(f"Final shape: {df.shape}")

---
## Summary of Cleaning Decisions

| Decision | Rationale |
|---|---|
| Replace `none`, `null`, `0`, `''` with `NaN` | Multiple inconsistent null representations — standardised to Pandas NaN |
| Drop rows missing First Name, Last Name, or Email | Cannot anonymise or uniquely identify the record without these fields |
| Remove exact duplicate rows | Duplicates would skew any downstream analysis |
| Replace names with SHA-256 hash of email | Reproducible unique ID; non-reversible; satisfies GDPR Art. 5(1)(c) data minimisation |
| Drop First Name, Last Name, Email columns | Direct identifiers removed as part of anonymisation |

**Note on `0` in numeric columns (e.g., Income):** A value of `0` in Income was treated as a null placeholder since legitimate zero income would be unusual in this context. In a real-world scenario this decision should be verified with the data owner.